# SingBERT — Singlish Hate Speech Detection Test

This notebook tests whether **SingBERT** (`zanelim/singbert`) can identify hate speech, offensive language, and normal speech in **Singlish** — without any additional fine-tuning.

### Approach
SingBERT is a masked language model (MLM), not a classifier. It has no classification head out-of-the-box. We test it two ways:

1. **Zero-shot via semantic similarity** — encode the input and compare its embedding to label description embeddings (`[CLS]` pooling + cosine similarity)
2. **Side-by-side with a standard English hate speech classifier** (`Hate-speech-CNERG/dehatebert-mono-english`) to see where the models agree/disagree on Singlish input

### Labels
- `hate speech` — targets a person/group based on identity
- `offensive` — rude or inappropriate, but not targeted hate
- `normal` — neutral or benign language

---

## Cell 1 — Install Dependencies

In [1]:
# Run once to install required packages
!pip install transformers torch scipy --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.57.0 requires numpy<1.25,>=1.21, but you have numpy 2.4.3 which is incompatible.
tables 3.8.0 requires blosc2~=2.0.0, but you have blosc2 3.0.0 which is incompatible.


## Cell 2 — Load Models

We load two models:
- **SingBERT** — Singlish/Manglish-aware BERT (encoder only, no classifier head)
- **dehatebert-mono** — English hate speech classifier (BERT fine-tuned on hate speech data)

First load will download model weights (~440MB for SingBERT, ~400MB for dehatebert). Subsequent runs use the local cache.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

singbert_tokenizer = BertTokenizer.from_pretrained('zanelim/singbert')
singbert_model = BertForSequenceClassification.from_pretrained(
    'zanelim/singbert',
    num_labels=3,
    id2label={0: "normal", 1: "offensive", 2: "hate speech"},
    label2id={"normal": 0, "offensive": 1, "hate speech": 2}
)
singbert_model.eval()
print("✅ SingBERT with classification head loaded")

print("\nLoading dehatebert-mono (English hate speech classifier)...")
# This model outputs: ['hate', 'noHate']
dehatebert = pipeline(
    'text-classification',
    model='Hate-speech-CNERG/dehatebert-mono-english',
    top_k=None  # return all labels with scores
)
print("✅ dehatebert-mono loaded")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\keith\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\keith\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\keith\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 711, in start
    self.io_loop.start()
  File "c:\Users\keith\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\keith\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\keith\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "c:\Users\keith\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 711, in start
    self.io_loop.start()
  File "c:\Users\keith\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found

Loading SingBERT tokenizer and model...


tokenizer_config.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

... (15 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


✅ dehatebert-mono loaded


## Cell 3 — Define Helper Functions

In [ ]:
def singbert_classify(text):
    """
    Run text through SingBERT classification head.
    Returns dict of {label: score}.
    NOTE: weights are random until fine-tuned — outputs will be unreliable.
    """
    inputs = singbert_tokenizer(
        text, return_tensors='pt', truncation=True, max_length=512
    )
    with torch.no_grad():
        outputs = singbert_model(**inputs)
    
    # outputs.logits shape: (1, 3) — one score per label
    probs = torch.softmax(outputs.logits, dim=1).squeeze()
    
    scores = {
        singbert_model.config.id2label[i]: probs[i].item()
        for i in range(len(probs))
    }
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))


def dehatebert_classify(text):
    """
    Run text through the dehatebert hate/noHate classifier.
    Returns simplified label and confidence.
    """
    results = dehatebert(text)[0]  # list of {'label': ..., 'score': ...}
    scores = {r['label']: r['score'] for r in results}
    top_label = max(scores, key=scores.get)
    return top_label, scores


LABEL_EMOJI = {"hate speech": "🚨", "offensive": "⚠️", "normal": "✅"}
HATE_EMOJI   = {"hate": "🚨", "noHate": "✅"}


def run_inference(text):
    """Run both models on a text and print a formatted comparison."""
    print(f"\n{'='*60}")
    print(f'📝 Input: "{text}"')
    print('='*60)

    # --- SingBERT zero-shot ---
    singbert_scores = singbert_classify(text)
    top_singbert_label = list(singbert_scores.keys())[0]
    print(f"\n🟡 SingBERT (zero-shot cosine similarity):")
    for label, score in singbert_scores.items():
        bar = '█' * int(score * 30)
        print(f"  {LABEL_EMOJI.get(label,'  ')} {label:<15} {score:.3f}  {bar}")
    print(f"  → Prediction: {LABEL_EMOJI.get(top_singbert_label,'')} {top_singbert_label.upper()}")

    # --- dehatebert ---
    dh_label, dh_scores = dehatebert_classify(text)
    print(f"\n🔵 dehatebert-mono (fine-tuned English classifier):")
    for label, score in sorted(dh_scores.items(), key=lambda x: x[1], reverse=True):
        bar = '█' * int(score * 30)
        print(f"  {HATE_EMOJI.get(label,'  ')} {label:<10} {score:.3f}  {bar}")
    print(f"  → Prediction: {HATE_EMOJI.get(dh_label,'')} {dh_label.upper()}")

    # --- Agreement check ---
    singbert_is_hate = top_singbert_label == "hate speech"
    dh_is_hate = dh_label == "hate"
    if singbert_is_hate == dh_is_hate:
        print("\n  🤝 Both models AGREE")
    else:
        print("\n  ⚡ Models DISAGREE — possible Singlish-specific gap")
    print()


print("\n✅ Helper functions ready. Proceed to Cell 4.")

Pre-computing label anchor embeddings from SingBERT...
  ✅ 'hate speech'
  ✅ 'offensive'
  ✅ 'normal'

✅ Helper functions ready. Proceed to Cell 4.


## Cell 4 — Interactive: Enter Your Own Text

Type any Singlish (or English) phrase and both models will classify it.

In [8]:
# ✏️ Enter your own text below — change this string and re-run the cell
user_input = input("Enter a sentence to classify (or press Enter for example): ").strip()

if not user_input:
    user_input = "wah lao eh, this fella damn suay sia"
    print(f'Using example: "{user_input}"')

run_inference(user_input)


📝 Input: "these immigrants need to go back to their dirty slums"

🟡 SingBERT (zero-shot cosine similarity):
  ✅ normal          0.336  ██████████
  ⚠️ offensive       0.332  █████████
  🚨 hate speech     0.332  █████████
  → Prediction: ✅ NORMAL

🔵 dehatebert-mono (fine-tuned English classifier):
     HATE       0.903  ███████████████████████████
     NON_HATE   0.097  ██
  → Prediction:  HATE

  🤝 Both models AGREE



## Cell 5 — Batch Test: Singlish Examples

Run a pre-defined set of Singlish test cases covering all three categories.

> **Note:** Some examples contain offensive language for research/testing purposes only.

In [9]:
# Test cases: (text, expected_label, notes)
test_cases = [
    # --- NORMAL ---
    ("wah lao eh, the mee pok today damn shiok leh",         "normal",     "casual food comment"),
    ("eh bro, later we go makan together can?",              "normal",     "friendly invite"),
    ("alamak, forgot to buy tissue again",                   "normal",     "mild frustration"),
    ("confirm plus chop, this show is sibei good",           "normal",     "positive recommendation"),
    ("ai yo, so leceh only",                                 "normal",     "mild complaint"),

    # --- OFFENSIVE ---
    ("nabeh, you damn stupid lah",                           "offensive",  "common Singlish expletive"),
    ("wah lan eh, you spoil my mood",                        "offensive",  "expletive, personal but not targeted hate"),
    ("cb, you sabo me again",                                "offensive",  "crude insult"),
    ("this fella bo liao one, sibei irritating",             "offensive",  "personal insult"),

    # --- HATE SPEECH (targeted based on identity) ---
    ("those ah tiong always come here take our jobs",        "hate speech", "xenophobia toward Chinese nationals"),
    ("all the bangla workers dirty and smelly one",          "hate speech", "ethnic slur toward South Asian workers"),
    ("malays always so lazy, that's why poor",               "hate speech", "racial stereotype"),
    ("india people cannot be trusted one",                   "hate speech", "racial generalisation"),
]

print("Running batch test on Singlish test cases...")
print("(Columns: Expected | SingBERT prediction | dehatebert prediction)\n")

# Summary table
results_table = []

for text, expected, notes in test_cases:
    sb_scores = singbert_zero_shot(text)
    sb_pred = list(sb_scores.keys())[0]
    dh_label, _ = dehatebert_classify(text)

    # Map dehatebert binary to our 3-class for display
    dh_simple = "hate speech" if dh_label == "hate" else "not hate"

    sb_correct = "✅" if sb_pred == expected else "❌"
    results_table.append((text[:45].ljust(46), expected.ljust(12), sb_pred.ljust(12), sb_correct, dh_simple, notes))

# Print header
print(f"{'Text':<46} {'Expected':<12} {'SingBERT':<12} {'OK?':<4} {'dehatebert':<12} Notes")
print('-' * 120)
for row in results_table:
    text, expected, sb_pred, sb_correct, dh_simple, notes = row
    print(f"{text} {expected} {sb_pred} {sb_correct}   {dh_simple:<12} {notes}")

# Accuracy
correct = sum(1 for r in results_table if r[3] == '✅')
total = len(results_table)
print(f"\n📊 SingBERT zero-shot accuracy: {correct}/{total} ({correct/total*100:.0f}%)")
print("(Note: SingBERT zero-shot is exploratory — low accuracy is expected without fine-tuning)")

Running batch test on Singlish test cases...
(Columns: Expected | SingBERT prediction | dehatebert prediction)

Text                                           Expected     SingBERT     OK?  dehatebert   Notes
------------------------------------------------------------------------------------------------------------------------
wah lao eh, the mee pok today damn shiok leh   normal       normal       ✅   not hate     casual food comment
eh bro, later we go makan together can?        normal       normal       ✅   not hate     friendly invite
alamak, forgot to buy tissue again             normal       normal       ✅   not hate     mild frustration
confirm plus chop, this show is sibei good     normal       normal       ✅   not hate     positive recommendation
ai yo, so leceh only                           normal       normal       ✅   not hate     mild complaint
nabeh, you damn stupid lah                     offensive    normal       ❌   not hate     common Singlish expletive
wah lan eh, 

## Cell 6 — Deeper Analysis: SingBERT Tokenisation

One thing to check: does SingBERT's tokeniser actually recognise Singlish words, or does it break them into subword fragments (which would indicate the vocabulary doesn't cover them well)?

In [10]:
singlish_words = [
    "lah", "leh", "lor", "meh", "sia",          # particles
    "wah", "alamak", "aiyo", "shiok", "bo liao",  # expressions
    "makan", "lepak", "sabo", "kiasu", "kena",    # common verbs/adj
    "ah tiong", "bangla", "ang moh",              # ethno-slang (may be sensitive)
    "nabeh", "wah lan", "cb",                      # expletives
]

print("🔍 Tokenisation check — does SingBERT know these Singlish words?")
print(f"{'Word':<15} {'Tokens':<40} {'Single token?'}")
print('-' * 65)

for word in singlish_words:
    tokens = singbert_tokenizer.tokenize(word)
    is_known = "✅ yes" if len(tokens) == 1 and not tokens[0].startswith('##') else f"⚠️  split into {len(tokens)}"
    print(f"{word:<15} {str(tokens):<40} {is_known}")

print("\n💡 Words split into multiple ##subword## tokens are not in the model's vocabulary")
print("   as single units — the model learned them from context, not as atomic tokens.")

🔍 Tokenisation check — does SingBERT know these Singlish words?
Word            Tokens                                   Single token?
-----------------------------------------------------------------
lah             ['lah']                                  ✅ yes
leh             ['leh']                                  ✅ yes
lor             ['lor']                                  ✅ yes
meh             ['meh']                                  ✅ yes
sia             ['sia']                                  ✅ yes
wah             ['wah']                                  ✅ yes
alamak          ['alam', '##ak']                         ⚠️  split into 2
aiyo            ['ai', '##yo']                           ⚠️  split into 2
shiok           ['shi', '##ok']                          ⚠️  split into 2
bo liao         ['bo', 'liao']                           ⚠️  split into 2
makan           ['ma', '##kan']                          ⚠️  split into 2
lepak           ['le', '##pa', '##k']              

## Cell 7 — Observations & Next Steps

Run this cell to print a summary of what this test tells us.

In [ ]:
print("""
=============================================================
OBSERVATIONS
=============================================================

SingBERT (zanelim/singbert) is a *language model*, not a
classifier. Using it zero-shot via cosine similarity against
label descriptions is a rough proxy — it tests whether the
model's contextual embeddings *happen* to group Singlish hate
speech closer to a 'hate speech description' embedding.

Expected findings:
  • SingBERT likely struggles at zero-shot hate detection
    because it was never trained to classify, only to predict
    masked tokens.
  • dehatebert-mono may misclassify Singlish expletives (e.g.
    'nabeh', 'wah lan') as hate speech or miss targeted slurs
    it hasn't seen before.
  • Tokenisation check (Cell 6) shows which Singlish words
    exist as single vocabulary items vs. subword fragments.

=============================================================
NEXT STEPS (to actually make this work well)
=============================================================

1. Fine-tune SingBERT on a labelled hate speech dataset:
   - Add a classification head on top of SingBERT
   - Label ~1,000–2,000 Singlish sentences
   - Fine-tune for text classification

2. Use SingBERT embeddings + a lightweight classifier:
   - Extract [CLS] embeddings from SingBERT
   - Train a logistic regression / SVM on top
   - Requires labelled data but less compute than full fine-tune

3. Consider the HuggingFace Trainer API:
   from transformers import BertForSequenceClassification, Trainer
   model = BertForSequenceClassification.from_pretrained(
       'zanelim/singbert', num_labels=3
   )
   # labels: 0=normal, 1=offensive, 2=hate speech

=============================================================
""")